# 04 · Medic'AM: extraction of reimbursed drugs linked to each campaign

Medic'AM (Assurance Maladie) gives, for every drug presentation (CIP13 code) dispensed
by community pharmacies and reimbursed by all health insurance schemes, the monthly
number of reimbursed boxes, the reimbursement base and the amount reimbursed, since
January 2015. Each presentation carries its ATC code (Anatomical Therapeutic Chemical
classification), which identifies the active substance.

This notebook:
1. downloads the monthly files from data.gouv.fr (or uses files you place by hand);
2. checks their layout;
3. extracts the drugs of each campaign (ATC codes below) as a tidy monthly table;
4. plots the series and decomposes them (STL) into trend, seasonality and remainder;
5. compares their long-run trend with Google Trends, if the monthly GT series exists.

**Why this part is not in the paper.** The selected drugs are treatments, the last step
of the cascade awareness → information seeking → screening → diagnosis → care. Their
dynamics are dominated by long-run trends (ageing, new therapies, reimbursement rules),
so campaign effects are not visible at the monthly scale. The notebook is kept as a
starting point for campaigns whose prescriptions are more seasonal (Mois sans tabac,
for example) and for linking search interest to downstream outcomes.

> The parser was tested on synthetic files reproducing the layout described by a public
> Medic'AM parser (header row with `CIP13`, `Code ATC` or `CODE 6`, then one column per
> month such as `Nombre de boîtes remboursées 2017-07`, and summary rows at the bottom).
> Run `inspect_file` (section 2) on the real files before trusting the output.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "phc").exists())
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt

from phc.config import CAMPAIGNS, ORDER, MEDICAM_RAW, MEDICAM_DIR, MONTHLY_DIR, FIG_DIR, TAB_DIR
from phc import medicam as M

DOWNLOAD = False         # True: download the monthly files from data.gouv.fr
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3})

## ATC codes per campaign

Specific drugs of the condition targeted by each campaign. Regorafenib appears twice
because its ATC code changed in 2022 (L01XE21 → L01EX05); older files use the old code.

In [ ]:
pd.DataFrame([{"campaign": CAMPAIGNS[c].label, "ATC": a, "molecule": m}
              for c in ORDER for a, m in M.ATC[c].items()])

## 1. Get the files

The dataset "Medic'AM : médicaments remboursés par l'Assurance Maladie, par type de
prescripteur (données interrégimes)" is published on data.gouv.fr, with one Excel file
per half-year for monthly data and annual files.

* **Automatic:** set `DOWNLOAD = True`. The cell lists the files of the dataset; check
  the titles, then adjust `PATTERN` so that only the monthly files are kept.
* **By hand:** download the half-year monthly files from the Assurance Maladie page
  "Médicaments délivrés par les pharmacies de ville par type de prescripteur –
  Medic'AM" and put them in `data/raw/medicam/`.

In [ ]:
PATTERN = r"mensuel"        # regular expression applied to the file titles

if DOWNLOAD:
    res = M.list_resources()
    display(res)
    selected = res[res["title"].str.contains(PATTERN, case=False, regex=True)
                   & res["format"].str.lower().isin(["xls", "xlsx"])]
    print(f"{len(selected)} files selected")
    M.download(selected, MEDICAM_RAW)

files = sorted(p for p in MEDICAM_RAW.glob("*") if p.suffix.lower() in (".xls", ".xlsx"))
print(f"{len(files)} Excel files in {MEDICAM_RAW.relative_to(ROOT)}")

## 2. Check the layout

For each file and sheet: the row where the header (`CIP13`) was found, the number of
monthly columns and the months covered. Files may contain several sheets (notes,
totals, prescriber types). Choose the sheet with the totals you want and set `SHEET`.

In [ ]:
if files:
    layout = pd.concat([M.inspect_file(p) for p in files], ignore_index=True)
    display(layout.drop(columns="columns"))
    print("Columns of the first sheet with a header:\n",
          layout.dropna(subset=["header_row"])["columns"].iloc[0])
else:
    print("No file yet: see section 1.")

In [ ]:
SHEET = 1      # index (0 = first sheet) or name of the sheet to read in every file

## 3. Extract the campaign drugs

Output: one row per presentation × month × measure, saved to
`data/processed/medicam/medicam_long.csv`, plus monthly totals per ATC code and per
campaign (number of reimbursed boxes).

In [ ]:
if files:
    codes = [a for c in ORDER for a in M.ATC[c]]
    long = M.parse_folder(MEDICAM_RAW, codes, sheet=SHEET)
    long.to_csv(MEDICAM_DIR / "medicam_long.csv", index=False)

    by_atc = M.monthly_by_atc(long, "boxes")
    by_campaign = M.monthly_by_campaign(long, "boxes")
    by_atc.to_csv(MEDICAM_DIR / "medicam_boxes_by_atc.csv")
    by_campaign.to_csv(MEDICAM_DIR / "medicam_boxes_by_campaign.csv")

    missing = [a for a in codes if a not in by_atc.columns]
    print(f"{long['cip13'].nunique()} presentations, "
          f"{by_campaign.index.min():%Y-%m} → {by_campaign.index.max():%Y-%m}")
    print("ATC codes not found in the files:", missing or "none")
    display(by_campaign.tail())

Gaps between files (a missing half-year) show up as missing months; check that the
index has no hole before decomposing the series.

In [ ]:
if files:
    full = pd.date_range(by_campaign.index.min(), by_campaign.index.max(), freq="MS")
    holes = full.difference(by_campaign.index)
    print("Missing months:", list(holes.strftime("%Y-%m")) or "none")

## 4. Monthly series and STL decomposition

In [ ]:
if files:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    for ax, c in zip(axes.flat, ORDER):
        cols = [a for a in M.ATC[c] if a in by_atc]
        for a in cols:
            ax.plot(by_atc.index, by_atc[a] / 1e3, lw=1, label=f"{a} {M.ATC[c][a]}")
        ax.set(title=CAMPAIGNS[c].label, ylabel="Reimbursed boxes (thousands)")
        ax.legend(fontsize=7)
    fig.suptitle("Medic'AM — reimbursed boxes per month, by molecule")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "medicam_boxes_by_molecule.png", dpi=200, bbox_inches="tight")

STL (Seasonal-Trend decomposition using LOESS) writes $y_t = T_t + S_t + R_t$ with a
12-month seasonal component. The table gives the share of each component in the total
variance of the components; `cov_share` is the part of var(y) due to covariance between
components. A large trend share and a small seasonal share mean that campaign months
do not stand out.

In [ ]:
if files and holes.empty:
    shares, decomp = {}, {}
    for c in ORDER:
        s = by_campaign[c].dropna()
        if len(s) >= 36:
            shares[CAMPAIGNS[c].label], decomp[c] = M.stl_shares(s)
    shares = pd.DataFrame(shares).T.round(1)
    shares.to_csv(TAB_DIR / "medicam_stl_variance_shares.csv")
    display(shares)

    fig, axes = plt.subplots(4, len(decomp), figsize=(4 * len(decomp), 9), sharex=True)
    for j, (c, res) in enumerate(decomp.items()):
        for i, (name, comp) in enumerate([("observed", res.observed), ("trend", res.trend),
                                          ("seasonal", res.seasonal), ("remainder", res.resid)]):
            ax = axes[i, j] if len(decomp) > 1 else axes[i]
            ax.plot(comp.index, comp / 1e3, lw=1, color=CAMPAIGNS[c].color)
            if i == 0:
                ax.set_title(CAMPAIGNS[c].label)
            if j == 0:
                ax.set_ylabel(f"{name}\n(thousand boxes)")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "medicam_stl.png", dpi=200, bbox_inches="tight")
elif files:
    print("Fill the missing months before running STL.")

## 5. Long-run trends: Medic'AM vs Google Trends

Both STL trends are rescaled to a maximum of 1 to compare their shapes. Needs
`data/processed/monthly/GT_monthly.csv` (notebook 01, section 4).

In [ ]:
gt_path = MONTHLY_DIR / "GT_monthly.csv"
if files and holes.empty and gt_path.exists():
    from statsmodels.tsa.seasonal import STL
    gt_m = pd.read_csv(gt_path, parse_dates=["date"])
    fig, axes = plt.subplots(2, 2, figsize=(13, 7))
    for ax, c in zip(axes.flat, ORDER):
        if c not in decomp:
            ax.set_visible(False)
            continue
        g = gt_m[gt_m["code"] == c].set_index("date")["value"].asfreq("MS")
        gt_trend = STL(g, period=12, robust=True).fit().trend
        med_trend = decomp[c].trend
        ax.plot(med_trend.index, med_trend / med_trend.max(), label="Medic'AM", color="#3a9a5b")
        ax.plot(gt_trend.index, gt_trend / gt_trend.max(), label="Google Trends", color="#2a6fb0")
        ax.set(title=CAMPAIGNS[c].label, ylabel="Trend (max = 1)")
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "medicam_vs_gt_trends.png", dpi=200, bbox_inches="tight")
elif files:
    print("Monthly Google Trends series not found: run notebook 01, section 4.")